In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df=spark.read.format('parquet')\
    .load('abfss://bronze@projecte2e.dfs.core.windows.net/orders')

In [0]:
display(df)

In [0]:
df.printSchema()

# **Drop unnecessary column**

In [0]:
df=df.drop("_rescued_data")
df.display()

**create a new year column**

In [0]:
from pyspark.sql.functions import *
df=df.withColumn("year",year(col('order_date')))

df.display()

## **Use of window function**

In [0]:
df1=df.withColumn('flag',dense_rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
df1.display()

In [0]:
df1=df.withColumn('rank_flag',rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
df1.display()

## **OOps_class**

In [0]:
class windows:
    def dense_rank(self,df):
        df_dense_rank=df.withColumn('dense_rank',dense_rank(),over(Window.partitionBy('year').orderBy(desc("total_amount"))))
        return dense_rank
    
    def rank_col(self,df):
        df_rank=df.withColumn('rank',rank().over(Window.partitionBy('year').orderBy(desc("total_amount"))))
        return rank_col
    
    def row_num(self,df):
        df_row_num=df.withColumn('row_num_col',row_number().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_row_num


In [0]:
df_new=df

In [0]:
obj=windows()

In [0]:
row_flag=obj.row_num(df_new)
row_flag.display()

### Data **Writting**

In [0]:
df.write.format('delta').mode('overwrite').save('abfss://silver@projecte2e.dfs.core.windows.net/orders')

In [0]:
%sql
create table if not exists project_cata.silver.orders_silver
using delta
location "abfss://silver@projecte2e.dfs.core.windows.net/orders"

In [0]:
%sql
select * from project_cata.silver.orders_silver